In [1]:
import pandas as pd
from global_macro_data import gmd

In [2]:
class GMDClient:
    def __init__(self):
        pass
    def get_gmd(self):
        return gmd(show_preview=False)
    def validate_type(self, gmd_df):
        if not isinstance(gmd_df, pd.DataFrame):
            raise TypeError(f"GMDClient expected pandas.DataFrame got {type(gmd_df)}")
        if gmd_df.empty:
            raise ValueError(f"GMD Dataframe ist empty")
        return gmd_df
    def run(self):
            df = self.get_gmd()
            return self.validate_type(df)
    
GMDClient().run().columns

Downloading: https://www.globalmacrodata.com/GMD_2025_12.csv
Final dataset: 56850 observations of 78 variables


Index(['countryname', 'ISO3', 'id', 'year', 'nGDP', 'nGDP_USD', 'rGDP',
       'rGDP_pc', 'rGDP_USD', 'deflator', 'cons', 'cons_GDP', 'cons_USD',
       'inv', 'inv_GDP', 'inv_USD', 'finv', 'finv_GDP', 'finv_USD', 'exports',
       'exports_GDP', 'exports_USD', 'imports', 'imports_GDP', 'imports_USD',
       'CA', 'CA_GDP', 'USDfx', 'REER', 'govexp', 'gen_govexp',
       'gen_govexp_GDP', 'cgovexp', 'cgovexp_GDP', 'govrev', 'gen_govrev',
       'cgovrev', 'gen_govrev_GDP', 'cgovrev_GDP', 'govtax', 'gen_govtax',
       'cgovtax', 'gen_govtax_GDP', 'cgovtax_GDP', 'govdef_GDP',
       'gen_govdef_GDP', 'gen_govdef', 'cgovdef_GDP', 'cgovdef', 'govdebt_GDP',
       'gen_govdebt_GDP', 'gen_govdebt', 'cgovdebt_GDP', 'cgovdebt', 'HPI',
       'CPI', 'infl', 'pop', 'unemp', 'strate', 'ltrate', 'cbrate', 'M0', 'M1',
       'M2', 'M3', 'M4', 'SovDebtCrisis', 'CurrencyCrisis', 'BankingCrisis',
       'CA_USD', 'govdebt', 'govdef', 'govexp_GDP', 'govrev_GDP', 'govtax_GDP',
       'rGDP_pc_USD', 'in

In [3]:
class GMDTransformer:
    def __init__(self):
        pass
    def clean_id(self, df):
        df = df.drop(columns=['id'])
        return df
    def objects_to_int(self, df):
        income_map = {"Low income": 1, "Lower middle income": 2, "Upper middle income": 3, "High income": 4,}
        df["income_group_code"] = df["income_group"].map(income_map).astype("Int8")
        df = df.drop(columns=['income_group'])
        return df
    def to_long(self, df):
        id_vars = ["countryname", "ISO3", "year"]
        df_long = df.melt(id_vars=id_vars, var_name="Item_Description", value_name="Value")
        return df_long
    def run(self, df):
        df = self.clean_id(df)
        df = self.objects_to_int(df)
        df = self.to_long(df)
        return df

In [4]:
class GMDIngestor:
    def __init__(self):
        self.client = GMDClient()
        self.transformer = GMDTransformer()
    def run(self):
        df = self.client.run()
        df = self.transformer.run(df)
        return df

In [7]:
gmd_df = GMDIngestor().run()

Downloading: https://www.globalmacrodata.com/GMD_2025_12.csv
Final dataset: 56850 observations of 78 variables


In [ ]:
hub = DataHub()

hub.insert_gmd(gmd_df)



DuckDB verbunden: c:\Diversification\data\01_raw\yahoo\yahoo_finance.db


,countryname,iso3,year,item_description,value,ingested_at
0,Aruba,ABW,1960,nGDP,NaN,2026-03-11 08:49:18.152004
1,Aruba,ABW,1961,nGDP,NaN,2026-03-11 08:49:18.152004
2,Aruba,ABW,1962,nGDP,NaN,2026-03-11 08:49:18.152004
3,Aruba,ABW,1963,nGDP,NaN,2026-03-11 08:49:18.152004
4,Aruba,ABW,1964,nGDP,NaN,2026-03-11 08:49:18.152004
5,Aruba,ABW,1965,nGDP,NaN,2026-03-11 08:49:18.152004
6,Aruba,ABW,1966,nGDP,NaN,2026-03-11 08:49:18.152004
7,Aruba,ABW,1967,nGDP,NaN,2026-03-11 08:49:18.152004
8,Aruba,ABW,1968,nGDP,NaN,2026-03-11 08:49:18.152004
9,Aruba,ABW,1969,nGDP,NaN,2026-03-11 08:49:18.152004


In [11]:
hub.preview_data("bronze_gmd", limit=60)

,countryname,iso3,year,item_description,value,ingested_at
0,Aruba,ABW,1960,nGDP,NaN,2026-03-11 08:49:18.152004
1,Aruba,ABW,1961,nGDP,NaN,2026-03-11 08:49:18.152004
2,Aruba,ABW,1962,nGDP,NaN,2026-03-11 08:49:18.152004
3,Aruba,ABW,1963,nGDP,NaN,2026-03-11 08:49:18.152004
4,Aruba,ABW,1964,nGDP,NaN,2026-03-11 08:49:18.152004
5,Aruba,ABW,1965,nGDP,NaN,2026-03-11 08:49:18.152004
6,Aruba,ABW,1966,nGDP,NaN,2026-03-11 08:49:18.152004
7,Aruba,ABW,1967,nGDP,NaN,2026-03-11 08:49:18.152004
8,Aruba,ABW,1968,nGDP,NaN,2026-03-11 08:49:18.152004
9,Aruba,ABW,1969,nGDP,NaN,2026-03-11 08:49:18.152004


In [8]:
from pathlib import Path
import duckdb
import pandas as pd


class DataHub:
    def __init__(self, db_name="yahoo_finance.db"):
        # Pfad-Management (funktioniert in Scripts & Notebooks)
        base_path = Path.cwd()
        db_path = base_path.parent.parent / "data" / "01_raw" / "yahoo" / db_name
        db_path.parent.mkdir(parents=True, exist_ok=True)

        # Verbindung herstellen
        self.con = duckdb.connect(str(db_path))
        self._initialize_tables()
        print(f"DuckDB verbunden: {db_path}")

    def _initialize_tables(self):
        """Erstellt die Tabellenstruktur, falls sie noch nicht existiert."""

        # Yahoo / Financials
        self.con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_financials (
                ticker VARCHAR,
                date DATE,
                affiliation VARCHAR,
                item_description VARCHAR,
                value DOUBLE,
                ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_ticker_date
            ON bronze_financials (ticker, date);
        """)

        # Wikidata
        self.con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_wikidata (
                company_qid VARCHAR,
                item_description VARCHAR,
                value VARCHAR,
                ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_wikidata_qid
            ON bronze_wikidata (company_qid);
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_wikidata_qid_item
            ON bronze_wikidata (company_qid, item_description);
        """)

        # GMD
        self.con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_gmd (
                countryname VARCHAR,
                iso3 VARCHAR,
                year INTEGER,
                item_description VARCHAR,
                value DOUBLE,
                ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_gmd_iso3_year
            ON bronze_gmd (iso3, year);
        """)
        self.con.execute("""
            CREATE INDEX IF NOT EXISTS idx_gmd_item
            ON bronze_gmd (item_description);
        """)

    def insert_financials(self, df: pd.DataFrame):
        """Speichert Yahoo-Financials in die Datenbank."""
        if df is None or df.empty:
            return

        df = df.copy()

        try:
            self.con.execute("""
                INSERT INTO bronze_financials (
                    ticker, date, affiliation, item_description, value
                )
                SELECT
                    ticker, date, affiliation, item_description, value
                FROM df
            """)
        except Exception as e:
            print(f"Fehler beim Insert in bronze_financials: {e}")

    def insert_wikidata(self, df: pd.DataFrame):
        """
        Speichert Wikidata-DataFrame in die Datenbank.
        Erwartete Spalten: company / Company_QID, Item_Description, Value
        """
        if df is None or df.empty:
            return

        df = df.copy()

        # Flexible Behandlung der QID-Spalte
        if "Company_QID" not in df.columns:
            if "company" in df.columns:
                df["Company_QID"] = df["company"].astype(str)
            else:
                raise ValueError(
                    f"Wikidata DF braucht 'Company_QID' oder 'company'. Vorhanden: {list(df.columns)}"
                )

        required = {"Company_QID", "Item_Description", "Value"}
        if not required.issubset(df.columns):
            missing = required - set(df.columns)
            raise ValueError(
                f"Wikidata DF fehlt Spalten: {missing}. Vorhanden: {list(df.columns)}"
            )

        df["Company_QID"] = df["Company_QID"].astype(str)
        df["Item_Description"] = df["Item_Description"].astype(str)
        df["Value"] = df["Value"].astype(str)

        try:
            self.con.execute("""
                INSERT INTO bronze_wikidata (
                    company_qid, item_description, value
                )
                SELECT
                    Company_QID AS company_qid,
                    Item_Description AS item_description,
                    Value AS value
                FROM df
            """)
        except Exception as e:
            print(f"Fehler beim Insert in bronze_wikidata: {e}")

    def insert_gmd(self, df: pd.DataFrame):
        """
        Speichert GMD-Long-DataFrame in die Datenbank.

        Erwartete Spalten:
        countryname | ISO3 | year | Item_Description | Value
        """
        if df is None or df.empty:
            return

        df = df.copy()

        required = {"countryname", "ISO3", "year", "Item_Description", "Value"}
        if not required.issubset(df.columns):
            missing = required - set(df.columns)
            raise ValueError(
                f"GMD DF fehlt Spalten: {missing}. Vorhanden: {list(df.columns)}"
            )

        # Typen normalisieren
        df["countryname"] = df["countryname"].astype(str)
        df["ISO3"] = df["ISO3"].astype(str)
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        df["Item_Description"] = df["Item_Description"].astype(str)
        df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

        # Optional: Zeilen ohne year verwerfen
        df = df[df["year"].notna()].copy()
        df["year"] = df["year"].astype(int)

        try:
            self.con.execute("""
                INSERT INTO bronze_gmd (
                    countryname, iso3, year, item_description, value
                )
                SELECT
                    countryname,
                    ISO3 AS iso3,
                    year,
                    Item_Description AS item_description,
                    Value AS value
                FROM df
            """)
        except Exception as e:
            print(f"Fehler beim Insert in bronze_gmd: {e}")

    def preview_data(self, table="bronze_wikidata", limit=100):
        """Holt Einträge als DataFrame zur Kontrolle."""
        return self.con.execute(
            f"SELECT * FROM {table} LIMIT {int(limit)}"
        ).df()

    def get_summary_stats(self):
        """Gibt eine kleine Statistik über den Füllstand der DB aus."""
        return self.con.execute("""
            SELECT
                (SELECT COUNT(DISTINCT ticker) FROM bronze_financials) AS count_tickers,
                (SELECT COUNT(*) FROM bronze_financials) AS total_financial_rows,
                (SELECT COUNT(DISTINCT company_qid) FROM bronze_wikidata) AS count_company_qids,
                (SELECT COUNT(*) FROM bronze_wikidata) AS total_wikidata_rows,
                (SELECT COUNT(DISTINCT iso3) FROM bronze_gmd) AS count_gmd_countries,
                (SELECT COUNT(*) FROM bronze_gmd) AS total_gmd_rows
        """).df()

    def close(self):
        """Schließt die Verbindung sauber."""
        self.con.close()

    def clear_database(self):
        """Löscht alle Daten und Tabellen aus der DuckDB-Datenbank."""
        try:
            tables = self.con.execute("SHOW TABLES").fetchall()

            if not tables:
                print("Datenbank ist bereits leer.")
                return

            print(f"Lösche {len(tables)} Tabellen...")

            for (table_name,) in tables:
                self.con.execute(f"DROP TABLE IF EXISTS {table_name}")

            print("Datenbank wurde erfolgreich geleert.")

        except Exception as e:
            print(f"Fehler beim Leeren der Datenbank: {e}")